# Road Accident Severity Prediction - Example Usage

This notebook demonstrates how to use the Road Accident Prediction System.

## 1. Setup and Imports

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_generator import AccidentDataGenerator
from src.preprocessing import AccidentDataPreprocessor
from src.model_training import AccidentSeverityModels
from src.environmental_analytics import EnvironmentalAnalytics

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 2. Generate Sample Data

In [ ]:
# Generate a small dataset for demo
generator = AccidentDataGenerator(n_samples=1000)
df = generator.generate_dataset()

print(f"Generated {len(df)} accident records")
df.head()

## 3. Data Exploration

In [ ]:
# Basic statistics
print("Dataset Info:")
print(df.describe())

# Severity distribution
severity_counts = df['severity'].value_counts().sort_index()
plt.figure(figsize=(10, 6))
severity_counts.plot(kind='bar', color=['green', 'yellow', 'orange', 'red'])
plt.title('Accident Severity Distribution')
plt.xlabel('Severity Level')
plt.ylabel('Count')
plt.xticks(range(4), ['Minor', 'Moderate', 'Severe', 'Fatal'], rotation=0)
plt.show()

## 4. Weather Impact Analysis

In [ ]:
# Weather condition vs severity
weather_severity = df.groupby('weather_condition')['severity'].mean().sort_values(ascending=False)

plt.figure(figsize=(12, 6))
weather_severity.plot(kind='barh', color='coral')
plt.title('Average Severity by Weather Condition')
plt.xlabel('Average Severity')
plt.ylabel('Weather Condition')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Time Pattern Analysis

In [ ]:
# Accidents by hour
hourly_accidents = df.groupby('hour')['accident_id'].count()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Hourly distribution
hourly_accidents.plot(kind='bar', ax=ax1, color='steelblue')
ax1.set_title('Accidents by Hour of Day')
ax1.set_xlabel('Hour')
ax1.set_ylabel('Number of Accidents')
ax1.grid(axis='y', alpha=0.3)

# Severity by hour
hourly_severity = df.groupby('hour')['severity'].mean()
hourly_severity.plot(ax=ax2, color='red', linewidth=2, marker='o')
ax2.set_title('Average Severity by Hour')
ax2.set_xlabel('Hour')
ax2.set_ylabel('Average Severity')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Load and Test Pre-trained Models

In [ ]:
# Load models
import joblib
from tensorflow import keras

try:
    rf_model = joblib.load('../models/random_forest.pkl')
    xgb_model = joblib.load('../models/xgboost.pkl')
    nn_model = keras.models.load_model('../models/neural_network.h5')
    preprocessor = joblib.load('../models/preprocessor.pkl')
    print("✅ Models loaded successfully!")
except:
    print("❌ Models not found. Please run training first.")
    print("   Run: python src/model_training.py")

## 7. Make Sample Predictions

In [ ]:
# Create a sample accident scenario
sample_data = pd.DataFrame({
    'datetime': [pd.Timestamp.now()],
    'hour': [22],
    'day_of_week': [5],
    'month': [12],
    'weather_condition': ['Rain'],
    'temperature': [5.0],
    'precipitation': [10.0],
    'visibility': [200.0],
    'road_surface': ['Wet'],
    'road_type': ['Highway'],
    'lighting_condition': ['Dark (Unlit)'],
    'speed_limit': [100],
    'vehicle_speed': [120.0],
    'vehicle_type': ['Car'],
    'num_vehicles': [2],
    'location_type': ['Rural'],
    'traffic_density': [50.0],
    'driver_age': [22.0],
    'driver_experience': [2.0],
    'alcohol_involved': [1],
    'accident_id': [0]
})

print("Sample Accident Scenario:")
print("- Time: 22:00, Saturday, December")
print("- Weather: Rain, Low visibility (200m)")
print("- Road: Wet highway, Dark (unlit)")
print("- Vehicle: Speeding (120 in 100 zone)")
print("- Driver: Young (22), Inexperienced (2 years), Alcohol involved")
print("\nPredicted Severity: ...")

# Note: Full prediction would require preprocessing the data
# See app.py for complete prediction pipeline

## 8. Model Performance Comparison

In [ ]:
# Load model comparison results
try:
    comparison = pd.read_csv('../models/model_comparison.csv')
    print("Model Performance Comparison:")
    print(comparison)
    
    # Visualize
    comparison.set_index('Model').plot(kind='bar', figsize=(12, 6))
    plt.title('Model Performance Comparison')
    plt.ylabel('Score')
    plt.xticks(rotation=45)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()
except:
    print("Model comparison data not available yet.")

## 9. Feature Importance Analysis

In [ ]:
# Analyze feature importance from Random Forest
try:
    with open('../data/feature_names.txt', 'r') as f:
        feature_names = f.read().splitlines()
    
    rf_model = joblib.load('../models/random_forest.pkl')
    importances = rf_model.feature_importances_
    
    # Create dataframe
    feature_imp = pd.DataFrame({
        'feature': feature_names,
        'importance': importances
    }).sort_values('importance', ascending=False).head(15)
    
    # Plot
    plt.figure(figsize=(12, 8))
    plt.barh(feature_imp['feature'], feature_imp['importance'], color='teal')
    plt.xlabel('Importance')
    plt.title('Top 15 Most Important Features (Random Forest)')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
except:
    print("Feature importance data not available yet.")

## 10. Summary and Recommendations

In [ ]:
print("="*60)
print("KEY FINDINGS".center(60))
print("="*60)
print("\n1. HIGH-RISK CONDITIONS:")
print("   - Poor weather (Rain, Fog, Storm)")
print("   - Low visibility (< 200m)")
print("   - Wet/Icy road surfaces")
print("   - Night time with poor lighting")
print("   - Speeding violations")
print("\n2. VULNERABLE GROUPS:")
print("   - Young drivers (< 25 years)")
print("   - Inexperienced drivers (< 3 years)")
print("   - Motorcycle and bicycle riders")
print("\n3. CRITICAL FACTORS:")
print("   - Alcohol involvement significantly increases severity")
print("   - Multiple vehicle collisions are more severe")
print("   - Speed differential from limit is a major factor")
print("\n" + "="*60)
print("\nFor full analytics, see: data/analytics/")
print("For web interface: python app.py")